# Agent 4 — Create Databricks Workflow

Creates the Gold KPI validation workflow.

Workflow structure:
  Task 1: run_kpi_baseline_computation  (weekly, independent)
  Task 2: run_kpi_validation_agent      (daily)
  Task 3: run_ai_summarization          (depends on Task 2)
  Task 4: run_alert_dispatcher          (depends on Task 3)

Schedule: 08:00 AM UTC daily
          Runs 2 hours after Agent 2 (06:00 AM UTC)

In [0]:
import requests
import json

host  = spark.conf.get("spark.databricks.workspaceUrl")
token = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type" : "application/json"
}

EXISTING_CLUSTER_ID = "0220-160255-4reafcow"

ROOT      = "/Users/s819534@8451.com/data_sentinel_barrows"
PIPE_PATH = f"{ROOT}/agent4/pipeline"

BASELINE_PATH  = f"{PIPE_PATH}/01_kpi_baseline_computation"
VALIDATION_PATH= f"{PIPE_PATH}/02_kpi_validation_agent"
AI_SUMM_PATH   = f"{PIPE_PATH}/03_ai_summarization"
ALERT_DISP_PATH= f"{PIPE_PATH}/04_teams_alert_dispatcher"

NOTIFICATION_EMAIL = "s819534@8451.com"

print("=" * 60)
print("AGENT 4 — WORKFLOW CREATION")
print("=" * 60)
print(f"Cluster ID    : {EXISTING_CLUSTER_ID}")
print(f"Baseline path : {BASELINE_PATH}")
print(f"Validation    : {VALIDATION_PATH}")
print(f"AI summ       : {AI_SUMM_PATH}")
print(f"Alert disp    : {ALERT_DISP_PATH}")
print("=" * 60)

## Step 1: Verify Notebooks Exist

In [0]:
def verify_notebook(path: str) -> bool:
    response = requests.get(
        f"https://{host}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path}
    )
    exists = response.status_code == 200
    status = "EXISTS  ✅" if exists else "MISSING ❌"
    print(f"  {status}  {path}")
    return exists

print("Verifying notebooks...")
all_exist = all([
    verify_notebook(BASELINE_PATH),
    verify_notebook(VALIDATION_PATH),
    verify_notebook(AI_SUMM_PATH),
    verify_notebook(ALERT_DISP_PATH)
])

if not all_exist:
    raise Exception(
        "One or more notebooks missing. "
        "Create them before creating the workflow."
    )
print("\nAll notebooks verified.")

## Step 2: Create Agent 4 Workflow

In [0]:
workflow = {
    "name": "agent4_gold_kpi_validation",

    # ── Schedule: 08:00 AM UTC daily ──────────────────────────────────────
    # Runs 2 hours after Agent 2 (06:00 AM UTC)
    # Gives Agent 2 time to complete reconciliation
    # before KPI validation begins
    "schedule": {
        "quartz_cron_expression": "0 0 8 * * ?",
        "timezone_id"           : "UTC",
        "pause_status"          : "PAUSED"    # enable after testing
    },

    "email_notifications": {
        "on_failure"               : [NOTIFICATION_EMAIL],
        "on_success"               : [],
        "no_alert_for_skipped_runs": True
    },

    "max_concurrent_runs": 1,
    "timeout_seconds"    : 7200,

    "tags": {
        "team"   : "data-engineering",
        "project": "data-trust-layer",
        "agent"  : "agent4"
    },

    # ── Job Parameters ─────────────────────────────────────────────────────
    # Single source of truth for target catalog/schema. Exposed to every
    # notebook task below as widgets ("catalog" / "schema"), so promoting
    # the whole job to a new catalog/schema only requires editing these
    # two defaults instead of every notebook.
    "parameters": [
        {"name": "catalog", "default": "learning_dev"},
        {"name": "schema"  , "default": "s819534"}
    ],

    "tasks": [

        # ── Task 1: KPI Baseline Computation ──────────────────────────────
        # Runs first but only does real work once a week (or on demand).
        # On most daily runs it will find no baseline changes needed
        # and complete quickly via the MERGE (no-op on existing records).
        {
            "task_key"           : "run_kpi_baseline_computation",
            "description"        : (
                "Computes KPI baseline statistics from clean historical data. "
                "Uses fixed pilot dates. Fast on most runs via MERGE."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "notebook_task"      : {
                "notebook_path": BASELINE_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 1800,
            "max_retries"              : 1,
            "min_retry_interval_millis": 120000,
            "retry_on_timeout"         : False
        },

        # ── Task 2: KPI Validation Agent ──────────────────────────────────
        {
            "task_key"           : "run_kpi_validation_agent",
            "description"        : (
                "Validates Gold KPI values for the given business date. "
                "Runs formula, cross-KPI, anomaly, and range checks."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {"task_key": "run_kpi_baseline_computation"}
            ],
            "notebook_task"      : {
                "notebook_path"   : VALIDATION_PATH,
                "base_parameters" : {
                    "business_date"    : "",   # blank = yesterday
                    "gold_table_filter": ""    # blank = all tables
                },
                "source"          : "WORKSPACE"
            },
            "timeout_seconds"          : 3600,
            "max_retries"              : 1,
            "min_retry_interval_millis": 300000,
            "retry_on_timeout"         : False,
            "email_notifications"      : {
                "on_failure": [NOTIFICATION_EMAIL]
            }
        },

        # ── Task 3: AI Summarization ───────────────────────────────────────
        {
            "task_key"           : "run_ai_summarization",
            "description"        : (
                "Enriches PENDING KPI validation runs with AI-generated "
                "summaries classified as CALCULATION_DRIFT, FRESHNESS_LAG, "
                "SEMANTIC_MISMATCH, or UNEXPLAINED_ANOMALY."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {"task_key": "run_kpi_validation_agent"}
            ],
            "notebook_task"      : {
                "notebook_path": AI_SUMM_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 1800,
            "max_retries"              : 1,
            "min_retry_interval_millis": 120000,
            "retry_on_timeout"         : False
        },

        # ── Task 4: Teams Alert Dispatcher ────────────────────────────────
        {
            "task_key"           : "run_alert_dispatcher",
            "description"        : (
                "Sends Teams alerts for CRITICAL and WARNING "
                "KPI validation runs that have not yet been alerted."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {"task_key": "run_ai_summarization"}
            ],
            "notebook_task"      : {
                "notebook_path": ALERT_DISP_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 900,
            "max_retries"              : 2,
            "min_retry_interval_millis": 60000,
            "retry_on_timeout"         : False
        }
    ]
}

print("Creating Agent 4 workflow...")
response = requests.post(
    f"https://{host}/api/2.1/jobs/create",
    headers=headers,
    json=workflow
)

print(f"Status code : {response.status_code}")

if response.status_code == 200:
    job_id = response.json().get("job_id")
    print(f"\n✅ Workflow created.")
    print(f"   Job ID   : {job_id}")
    print(f"   View at  : https://{host}/#job/{job_id}")
    print()
    print("Schedule is PAUSED.")
    print("Run test cell below before enabling schedule.")
else:
    print(f"\n❌ Failed: {response.text}")
    job_id = None

## Step 3: Reset Test Data and Trigger Manual Test Run

In [0]:
if job_id:
    TEST_DATE = "2026-06-24"

    # Reset test date so workflow generates fresh results
    print(f"Resetting {TEST_DATE} results for clean test run...")
    spark.sql(f"""
        DELETE FROM learning_dev.s819534.uc4_kpi_validation_results_b
        WHERE business_date = '{TEST_DATE}'
    """)
    spark.sql(f"""
        DELETE FROM learning_dev.s819534.uc4_kpi_validation_run_b
        WHERE business_date = '{TEST_DATE}'
    """)
    print(f"Reset complete.")
    print()

    trigger_payload = {
        "job_id"          : int(job_id),
        "notebook_params" : {
            "business_date"    : TEST_DATE,
            "gold_table_filter": ""
        }
    }

    print(f"Triggering test run for {TEST_DATE}...")
    trigger_resp = requests.post(
        f"https://{host}/api/2.1/jobs/run-now",
        headers=headers,
        json=trigger_payload
    )

    print(f"Status code : {trigger_resp.status_code}")

    if trigger_resp.status_code == 200:
        run_id = trigger_resp.json().get("run_id")
        print(f"\n✅ Test run triggered.")
        print(f"   Run ID  : {run_id}")
        print(f"   View at : https://{host}/#job/{job_id}/run/{run_id}")
    else:
        print(f"\n❌ Trigger failed: {trigger_resp.text}")
        run_id = None

## Step 4: Monitor the Run

In [0]:
import time
from datetime import datetime

def monitor_run(run_id: int, poll_seconds: int = 20):
    if not run_id:
        print("No run_id. Trigger a run first.")
        return

    terminal_states = {"TERMINATED", "SKIPPED", "INTERNAL_ERROR"}

    task_emojis = {
        "run_kpi_baseline_computation": "📊",
        "run_kpi_validation_agent"    : "🔍",
        "run_ai_summarization"        : "🤖",
        "run_alert_dispatcher"        : "📣"
    }

    status_emojis = {
        "RUNNING"       : "⏳",
        "TERMINATED"    : "✅",
        "INTERNAL_ERROR": "❌",
        "PENDING"       : "⏸",
        "SKIPPED"       : "⏭"
    }

    print(f"Monitoring run {run_id}...")
    print()
    previous_states = {}

    while True:
        resp = requests.get(
            f"https://{host}/api/2.1/jobs/runs/get",
            headers=headers,
            params={"run_id": run_id}
        )

        if resp.status_code != 200:
            print(f"Polling error: {resp.status_code}")
            break

        run_data   = resp.json()
        state      = run_data.get("state", {})
        life_state = state.get("life_cycle_state", "UNKNOWN")
        result     = state.get("result_state", "")
        tasks      = run_data.get("tasks", [])

        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] Overall: {life_state} {result}")

        for task in tasks:
            task_key   = task.get("task_key", "")
            task_state = task.get("state", {})
            task_life  = task_state.get("life_cycle_state", "PENDING")
            task_result= task_state.get("result_state", "")
            t_emoji    = task_emojis.get(task_key, "⚙️")
            s_emoji    = status_emojis.get(task_life, "❓")

            current = f"{task_life} {task_result}"
            if previous_states.get(task_key) != current:
                previous_states[task_key] = current
                print(
                    f"  {t_emoji} {task_key:<35} "
                    f"{s_emoji} {task_life:<15} "
                    f"{task_result}"
                )

        print()

        if life_state in terminal_states:
            print("=" * 55)
            if result == "SUCCESS":
                print("✅ Agent 4 workflow completed successfully.")
            else:
                print(f"❌ Workflow ended: {result}")
                msg = state.get("state_message", "")
                if msg:
                    print(f"   Message: {msg}")
            print("=" * 55)
            break

        time.sleep(poll_seconds)


if run_id:
    monitor_run(int(run_id))

## Step 5: Verify End-to-End Results

In [0]:
if job_id and run_id:
    print(f"Verification after workflow run for {TEST_DATE}:\n")

    spark.sql(f"""
        SELECT
            business_date,
            SUBSTRING_INDEX(gold_fully_qualified_name, '.', -1) AS gold_table,
            overall_status,
            total_kpis_checked,
            total_kpis_passed,
            total_kpis_warned,
            total_kpis_critical,
            ai_enrichment_status,
            CASE WHEN alert_sent = true THEN 'YES' ELSE 'NO' END AS alert_fired
        FROM learning_dev.s819534.uc4_kpi_validation_run_b
        WHERE business_date = '{TEST_DATE}'
        ORDER BY gold_fully_qualified_name
    """).show(truncate=False)

## Step 6: Enable the Schedule (After Successful Test)

In [0]:
def enable_schedule(job_id: int):
    """
    Enables the daily 08:00 AM UTC schedule.
    Only run after a successful manual test.
    """
    response = requests.patch(
        f"https://{host}/api/2.1/jobs/{job_id}",
        headers=headers,
        json={
            "new_settings": {
                "schedule": {
                    "quartz_cron_expression": "0 0 8 * * ?",
                    "timezone_id"           : "UTC",
                    "pause_status"          : "UNPAUSED"
                }
            }
        }
    )

    if response.status_code == 200:
        print(f"✅ Schedule enabled.")
        print(f"   Agent 4 will run daily at 08:00 AM UTC.")
        print(f"   https://{host}/#job/{job_id}")
    else:
        print(f"❌ Failed: {response.text}")


# Uncomment after successful test
# enable_schedule(int(job_id))
print("Schedule is currently PAUSED.")
print("Uncomment enable_schedule() and rerun after successful test.")